## **Cell 1 — Import libraries**

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


### **Cell 2 — Upload dataset**

In [ ]:
# Reproducible data loading
# Use breast_cancer.csv when it is available; otherwise rebuild the same
# Wisconsin diagnostic dataset from scikit-learn so "Run all" works cleanly.
from pathlib import Path
import numpy as np
import pandas as pd

def load_project_data():
    candidates = [
        Path("breast_cancer.csv"),
        Path("/content/breast_cancer.csv"),
        Path("/mnt/data/breast_cancer.csv"),
    ]
    for path in candidates:
        if path.exists():
            data = pd.read_csv(path)
            data.columns = data.columns.str.strip()
            if {"id", "diagnosis"}.issubset(data.columns):
                print(f"Loaded: {path}")
                return data

    from sklearn.datasets import load_breast_cancer
    raw = load_breast_cancer(as_frame=True)
    features = raw.data.copy()

    def project_name(name):
        if name.startswith("mean "):
            return name[5:].replace(" ", "_") + "_mean"
        if name.endswith(" error"):
            return name[:-6].replace(" ", "_") + "_se"
        if name.startswith("worst "):
            return name[6:].replace(" ", "_") + "_worst"
        return name.replace(" ", "_")

    features.columns = [project_name(c) for c in features.columns]
    data = features.copy()
    data.insert(0, "id", np.arange(1, len(data) + 1))
    data.insert(1, "diagnosis", np.where(raw.target.to_numpy() == 0, "M", "B"))
    print("breast_cancer.csv not found; loaded the equivalent scikit-learn dataset.")
    return data

df = load_project_data()
print("Dataset shape:", df.shape)


In [ ]:
print(df.shape)
df.head()


## **Cell 3 — Understand dataset**

In [27]:
print(df.info())

print("\nDiagnosis:")
print(df["diagnosis"].value_counts())

print("\nPercentages:")
print(df["diagnosis"].value_counts(normalize=True) * 100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave_points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

### **Cell 4 — Data cleaning checks**


In [28]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

Missing values: 0
Duplicate rows: 0
Duplicate IDs: 0


In [29]:
numeric = df.select_dtypes(include=np.number)

print("Infinite values:", np.isinf(numeric).sum().sum())

Infinite values: 0


### **Cell 5 — Check outliers, but don't remove them**

In [30]:
feature_cols = [
    c for c in df.columns
    if c not in ["id", "diagnosis"]
]

Q1 = df[feature_cols].quantile(0.25)
Q3 = df[feature_cols].quantile(0.75)

IQR = Q3 - Q1

outliers = (
    (df[feature_cols] < (Q1 - 1.5 * IQR)) |
    (df[feature_cols] > (Q3 + 1.5 * IQR))
)

print("Number of flagged outlier values:", outliers.sum().sum())

Number of flagged outlier values: 608


### **Cell 6 — Understand Mean / SE / Worst**

In [31]:
mean_features = [c for c in feature_cols if c.endswith("_mean")]
se_features = [c for c in feature_cols if c.endswith("_se")]
worst_features = [c for c in feature_cols if c.endswith("_worst")]

print("Mean:", len(mean_features))
print("SE:", len(se_features))
print("Worst:", len(worst_features))

Mean: 10
SE: 10
Worst: 10


### **Cell 7 — Create X and target y**

In [32]:
X = df.drop(columns=["id", "diagnosis"])

y = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

print("X:", X.shape)
print("y:", y.shape)

print(y.value_counts())

X: (569, 30)
y: (569,)
diagnosis
0    357
1    212
Name: count, dtype: int64


## **Cell 8 — 80/20 stratified split**

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (455, 30)
Testing: (114, 30)


### **Cell 9 — Verify same B/M proportions**

In [34]:
print("Full dataset:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nTesting:")
print(y_test.value_counts(normalize=True))

Full dataset:
diagnosis
0    0.627417
1    0.372583
Name: proportion, dtype: float64

Training:
diagnosis
0    0.626374
1    0.373626
Name: proportion, dtype: float64

Testing:
diagnosis
0    0.631579
1    0.368421
Name: proportion, dtype: float64


### **Cell 10 — Save common datasets**

In [35]:
train_df = X_train.copy()
train_df["target"] = y_train

test_df = X_test.copy()
test_df["target"] = y_test

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Files saved.")

Files saved.


### **Cell 11 — Download them**

In [ ]:
print("Saved train.csv in the current runtime.")


In [ ]:
print("Saved test.csv in the current runtime.")
